In [1]:
from data_collector import *
import pandas as pd
from config import GOOGLE_MAPS_DATASETS
pd.set_option("display.max.columns", None)

# Finding Aggregated Places in CPH Bounding box

As as 'sanity check', we use Aggregated Places to compare the result to the Quadtree implementation.

In [2]:
COPENHAGEN_BOUNDS = {
    "lat_south": 55.51,
    "lat_north": 55.82,
    "lon_west": 12.23,
    "lon_east": 12.73,
}

response = find_aggregated_places(COPENHAGEN_BOUNDS) 
cph_df = pd.read_json(GOOGLE_MAPS_DATASETS / "copenhagen-bounds-limit10_20260316_125502.json")

print(f"Shape of Quadtree implementation: {cph_df.shape}")
print(f"Response from Places Aggregated with same BB: {response}")

Shape of Quadtree implementation: (4392, 66)
Response from Places Aggregated with same BB: {'error': {'code': 403, 'message': 'This API method requires billing to be enabled. Please enable billing on project #357676775000 by visiting https://console.developers.google.com/billing/enable?project=357676775000 then retry. If you enabled billing for this project recently, wait a few minutes for the action to propagate to our systems and retry.', 'status': 'PERMISSION_DENIED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'BILLING_DISABLED', 'domain': 'googleapis.com', 'metadata': {'consoleUrl': 'https://console.developers.google.com/billing/enable?project=357676775000', 'service': 'areainsights.googleapis.com', 'containerInfo': '357676775000', 'consumer': 'projects/357676775000'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'This API method requires billing to be enabled. Please enable billing on project #35767677

# Exploring the Copenhagen Data 

In [3]:
cph_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4392 entries, 0 to 4391
Data columns (total 66 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   name                          4392 non-null   str    
 1   id                            4392 non-null   str    
 2   types                         4392 non-null   object 
 3   nationalPhoneNumber           3758 non-null   str    
 4   internationalPhoneNumber      3758 non-null   str    
 5   formattedAddress              4392 non-null   str    
 6   addressComponents             4392 non-null   object 
 7   plusCode                      4381 non-null   object 
 8   location                      4392 non-null   object 
 9   viewport                      4392 non-null   object 
 10  rating                        4046 non-null   float64
 11  googleMapsUri                 4392 non-null   str    
 12  websiteUri                    3600 non-null   str    
 13  regularOpening

In [4]:
cph_df.describe()

,rating,utcOffsetMinutes,userRatingCount,takeout,delivery,dineIn,reservable,servesBreakfast,servesLunch,servesDinner,servesBeer,servesWine,servesBrunch,outdoorSeating,liveMusic,menuForChildren,servesCocktails,servesDessert,servesCoffee,goodForChildren,allowsDogs,restroom,goodForGroups,goodForWatchingSports,curbsidePickup,servesVegetarianFood
count,4046.000000,4392.0,4046.000000,3237.000000,2968.000000,3910.000000,2367.000000,1732.000000,3009.000000,3249.000000,2754.000000,2451.000000,1713.000000,2187.000000,3465.000000,2044.000000,2035.000000,2428.000000,2573.000000,2214.000000,1481.000000,3003.000000,1847.000000,3126.000000,1205.000000,1497.000000
mean,4.197257,60.0,398.480969,0.914427,0.448787,0.993350,0.820870,0.368938,0.969757,0.978763,0.740015,0.722154,0.308815,0.718793,0.043001,0.692270,0.515971,0.871911,0.833657,0.803975,0.160702,0.876457,0.965891,0.014395,0.351037,0.981296
std,0.548070,0.0,869.449306,0.279776,0.497454,0.081284,0.383542,0.482656,0.171283,0.144197,0.438706,0.448028,0.462140,0.449691,0.202890,0.461667,0.499868,0.334258,0.372460,0.397078,0.367380,0.329114,0.181559,0.119133,0.477493,0.135523
min,1.000000,60.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.000000,60.0,52.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000
50%,4.300000,60.0,161.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000
75%,4.600000,60.0,435.750000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000
max,5.000000,60.0,25668.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
import json
from collections import Counter

with open((GOOGLE_MAPS_DATASETS / "copenhagen-bounds-limit10_20260316_125502.json"), "r", encoding="utf-8") as f:
    data = json.load(f)
# Extract locality from addressComponents
localities = []
for place in data:
    address_components = place.get("addressComponents", [])
    for component in address_components:
        if "locality" in component.get("types", []):
            localities.append(component["longText"])
            break

# Count and display
counter = Counter(localities)
for municipality, count in counter.most_common():
    print(f"{municipality}: {count}")

København: 2743
Frederiksberg: 331
Kongens Lyngby: 114
Kastrup: 97
Hvidovre: 87
Rødovre: 79
Taastrup: 75
Hellerup: 70
Greve: 62
Ballerup: 60
Charlottenlund: 60
Søborg: 54
Herlev: 49
Glostrup: 48
Klampenborg: 47
Gentofte: 41
Albertslund: 37
Ishøj: 37
Bagsværd: 30
Brøndby: 29
Dragør: 25
Værløse: 23
Farum: 23
Virum: 22
Holte: 21
Vallensbæk Strand: 17
Copenhagen: 16
Brøndby Strand: 13
Skovlunde: 10
Karlslunde: 10
Dyssegård: 9
Måløv: 8
Smørum: 6
Nærum: 6
Stenløse: 4
Vallensbæk: 3
Veksø: 2
Hovedstaden: 2
København K: 2
Amager: 2
Slangerup: 1
Veksø Sjælland: 1
Skodsborg: 1
Kgs Lyngby: 1
Skovshoved: 1
kbh k: 1
kl: 1
søsiden: 1
Solrød Strand: 1
Sundbyøster: 1
Fields: 1


In [6]:
cols_to_keep = [
    "name", 
    "id", 
    "types", 
    "formattedAddress", 
    "location", 
    "rating", 
    "googleMapsUri",
    "websiteUri",
    "businessStatus",
    "priceLevel",
    "displayName",
    "takeout",
    "delivery",
    "dineIn",
    "servesBreakfast",
    "servesLunch",
    "servesDinner",
    "servesBrunch",
    "servesDessert",
    "primaryType",
    "primaryTypeDisplayName",
    "reviews",
    "priceRange",
    "postalAddress",
    "editorialSummary",
    "servesVegetarianFood",
]

len(cols_to_keep)

26

In [7]:
cph_df = cph_df[cols_to_keep]
cph_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4392 entries, 0 to 4391
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   name                    4392 non-null   str    
 1   id                      4392 non-null   str    
 2   types                   4392 non-null   object 
 3   formattedAddress        4392 non-null   str    
 4   location                4392 non-null   object 
 5   rating                  4046 non-null   float64
 6   googleMapsUri           4392 non-null   str    
 7   websiteUri              3600 non-null   str    
 8   businessStatus          4392 non-null   str    
 9   priceLevel              1738 non-null   str    
 10  displayName             4392 non-null   object 
 11  takeout                 3237 non-null   float64
 12  delivery                2968 non-null   float64
 13  dineIn                  3910 non-null   float64
 14  servesBreakfast         1732 non-null   float64
 15

In [8]:
cph_df.describe()

,rating,takeout,delivery,dineIn,servesBreakfast,servesLunch,servesDinner,servesBrunch,servesDessert,servesVegetarianFood
count,4046.000000,3237.000000,2968.000000,3910.000000,1732.000000,3009.000000,3249.000000,1713.000000,2428.000000,1497.000000
mean,4.197257,0.914427,0.448787,0.993350,0.368938,0.969757,0.978763,0.308815,0.871911,0.981296
std,0.548070,0.279776,0.497454,0.081284,0.482656,0.171283,0.144197,0.462140,0.334258,0.135523
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.000000,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000
50%,4.300000,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000
75%,4.600000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [9]:
mini_box = pd.read_json(GOOGLE_MAPS_DATASETS / 'mini-box_20260311_101610.json')
mini_box.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 64 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   name                          31 non-null     str    
 1   id                            31 non-null     str    
 2   types                         31 non-null     object 
 3   formattedAddress              31 non-null     str    
 4   addressComponents             31 non-null     object 
 5   plusCode                      31 non-null     object 
 6   location                      31 non-null     object 
 7   viewport                      31 non-null     object 
 8   rating                        29 non-null     float64
 9   googleMapsUri                 31 non-null     str    
 10  websiteUri                    25 non-null     str    
 11  regularOpeningHours           28 non-null     object 
 12  utcOffsetMinutes              31 non-null     int64  
 13  adrFormatAddress  

In [10]:
noma = cph_df[cph_df['displayName'].apply(lambda x: x['text'] == "Mad & Kaffe")]

# import pandas as pd
# pd.set_option('display.max_colwidth', None)
# print(noma['priceRange'])

[name], 	[places/ChIJpYCQZztTUkYRFOE368Xs6kI],
[id], 	[ChIJpYCQZztTUkYRFOE368Xs6kI],
[types], 	[[scandinavian_restaurant, fine_dining_restaura...],
[formattedAddress], 	[Refshalevej 96, 1432 Indre By, Denmark],
[location], 	[{'latitude': 55.6828273, 'longitude': 12.6104808}],
[rating], 	[4.6],
[googleMapsUri], 	[https://maps.google.com/?cid=48219266858525575...],
[websiteUri], 	[https://noma.dk/],
[businessStatus], 	[CLOSED_TEMPORARILY],
[priceLevel], 	[PRICE_LEVEL_VERY_EXPENSIVE],
[displayName], 	[{'text': 'Noma', 'languageCode': 'en'}],
[takeout], 	[0.0],
[delivery], 	[0.0],
[dineIn], 	[1.0],
[servesBreakfast], 	[0.0],
[servesLunch], 	[1.0],
[servesDinner], 	[1.0],
[servesBrunch], 	[0.0],
[servesDessert], 	[1.0],
[primaryType], 	[scandinavian_restaurant],
[primaryTypeDisplayName], 	[{'text': 'Scandinavian Restaurant', 'languageC...],
[reviews], 	[[{'name': 'places/ChIJpYCQZztTUkYRFOE368Xs6kI/...],
[priceRange], 	[{'startPrice': {'currencyCode': 'DKK', 'units'...],
[postalAddress], 	[{'regionCode': 'DK', 'languageCode': 'en-US', ...],
[editorialSummary], 	[{'text': 'Chef Rene Redzepi's gastronomic mecc...],
[servesVegetarianFood], 	[1.0],


In [11]:
noma.T

,2615,4177
name,places/ChIJAcjGaXZTUkYRXFFcKCkRt0c,places/ChIJjYGh-kxTUkYRuO65MpgG4eA
id,ChIJAcjGaXZTUkYRXFFcKCkRt0c,ChIJjYGh-kxTUkYRuO65MpgG4eA
types,"[cafe, brunch_restaurant, breakfast_restaurant...","[cafe, brunch_restaurant, coffee_shop, breakfa..."
formattedAddress,"Sønder Blvd. 68, 1720 Vesterbro, Denmark","Tyrolsgade 6, 2300 Amager Øst, Denmark"
location,"{'latitude': 55.6659376, 'longitude': 12.5503967}","{'latitude': 55.659096299999995, 'longitude': ..."
rating,4.5,4.4
googleMapsUri,https://maps.google.com/?cid=51676179658994036...,https://maps.google.com/?cid=16204240185011596...
websiteUri,http://www.madogkaffe.dk/,http://www.madogkaffe.dk/
businessStatus,OPERATIONAL,OPERATIONAL
priceLevel,PRICE_LEVEL_MODERATE,PRICE_LEVEL_MODERATE
